In [1]:
import pandas as pd
import re
import string
import nltk
from nltk.corpus import stopwords
from Sastrawi.Stemmer.StemmerFactory import StemmerFactory
from tqdm import tqdm

tqdm.pandas()

In [2]:
pd.set_option('display.max_rows', 10)
pd.set_option('display.max_colwidth', 50)

# Load Data

In [3]:
df = pd.read_excel('dataset_merged_labeled.xlsx')
print(f"--- TOTAL DATA BERHASIL DI-LOAD: {len(df)} ---\n")
display(df[['text_original', 'label']])

--- TOTAL DATA BERHASIL DI-LOAD: 4155 ---



,text_original,label
0,Habiburokhman soal Polisi Tembak Siswa: Seenak...,0
1,video anak anak gaza mendapat kesempatan berma...,1
2,video hari ini ditemukan oleh relawan anies di...,1
3,anies bongkar misi busuk prabowo usai tom lemb...,1
4,video demo cirebon pada 1 juni 2024 Akun Tikto...,1
...,...,...
4150,video warga israel kabur dengan tangga eskalat...,1
4151,Puan Berkebaya Kuning Saat Rayakan Lebaran den...,0
4152,frasa agama akan dihapus dalam peta jalan pend...,1
4153,samsung menarik iklan senilai 1 miliar dengan ...,1


# Process Preprocessing

# Case Folding

In [6]:
df['step_1_folding'] = df['text_original'].str.lower()

In [7]:
print("\n--- STEP 1: CASE FOLDING SELESAI (Mengubah ke Huruf Kecil) ---")
display(df[['text_original', 'step_1_folding']])


--- STEP 1: CASE FOLDING SELESAI (Mengubah ke Huruf Kecil) ---


,text_original,step_1_folding
0,Habiburokhman soal Polisi Tembak Siswa: Seenak...,habiburokhman soal polisi tembak siswa: seenak...
1,video anak anak gaza mendapat kesempatan berma...,video anak anak gaza mendapat kesempatan berma...
2,video hari ini ditemukan oleh relawan anies di...,video hari ini ditemukan oleh relawan anies di...
3,anies bongkar misi busuk prabowo usai tom lemb...,anies bongkar misi busuk prabowo usai tom lemb...
4,video demo cirebon pada 1 juni 2024 Akun Tikto...,video demo cirebon pada 1 juni 2024 akun tikto...
...,...,...
4150,video warga israel kabur dengan tangga eskalat...,video warga israel kabur dengan tangga eskalat...
4151,Puan Berkebaya Kuning Saat Rayakan Lebaran den...,puan berkebaya kuning saat rayakan lebaran den...
4152,frasa agama akan dihapus dalam peta jalan pend...,frasa agama akan dihapus dalam peta jalan pend...
4153,samsung menarik iklan senilai 1 miliar dengan ...,samsung menarik iklan senilai 1 miliar dengan ...


# Cleaning

In [8]:
def remove_noise(text):
    text = re.sub(r'http\S+|www\S+|https\S+|@\S+|#\S+', '', text)
    text = text.translate(str.maketrans('', '', string.punctuation + string.digits))
    return text.strip()

In [9]:
df['step_2_cleaning'] = df['step_1_folding'].apply(remove_noise)
print("\n--- STEP 2: CLEANING SELESAI (Menghapus Simbol, Angka, & URL) ---")
display(df[['step_1_folding', 'step_2_cleaning']])


--- STEP 2: CLEANING SELESAI (Menghapus Simbol, Angka, & URL) ---


,step_1_folding,step_2_cleaning
0,habiburokhman soal polisi tembak siswa: seenak...,habiburokhman soal polisi tembak siswa seenakn...
1,video anak anak gaza mendapat kesempatan berma...,video anak anak gaza mendapat kesempatan berma...
2,video hari ini ditemukan oleh relawan anies di...,video hari ini ditemukan oleh relawan anies di...
3,anies bongkar misi busuk prabowo usai tom lemb...,anies bongkar misi busuk prabowo usai tom lemb...
4,video demo cirebon pada 1 juni 2024 akun tikto...,video demo cirebon pada juni akun tiktok rob...
...,...,...
4150,video warga israel kabur dengan tangga eskalat...,video warga israel kabur dengan tangga eskalat...
4151,puan berkebaya kuning saat rayakan lebaran den...,puan berkebaya kuning saat rayakan lebaran den...
4152,frasa agama akan dihapus dalam peta jalan pend...,frasa agama akan dihapus dalam peta jalan pend...
4153,samsung menarik iklan senilai 1 miliar dengan ...,samsung menarik iklan senilai miliar dengan o...


# Tokenize

In [10]:
df['step_3_tokenizing'] = df['step_2_cleaning'].apply(lambda x: x.split())

In [11]:
print("\n--- STEP 3: TOKENIZING SELESAI (Memecah Kalimat Menjadi Kata) ---")
display(df[['step_2_cleaning', 'step_3_tokenizing']])


--- STEP 3: TOKENIZING SELESAI (Memecah Kalimat Menjadi Kata) ---


,step_2_cleaning,step_3_tokenizing
0,habiburokhman soal polisi tembak siswa seenakn...,"[habiburokhman, soal, polisi, tembak, siswa, s..."
1,video anak anak gaza mendapat kesempatan berma...,"[video, anak, anak, gaza, mendapat, kesempatan..."
2,video hari ini ditemukan oleh relawan anies di...,"[video, hari, ini, ditemukan, oleh, relawan, a..."
3,anies bongkar misi busuk prabowo usai tom lemb...,"[anies, bongkar, misi, busuk, prabowo, usai, t..."
4,video demo cirebon pada juni akun tiktok rob...,"[video, demo, cirebon, pada, juni, akun, tikto..."
...,...,...
4150,video warga israel kabur dengan tangga eskalat...,"[video, warga, israel, kabur, dengan, tangga, ..."
4151,puan berkebaya kuning saat rayakan lebaran den...,"[puan, berkebaya, kuning, saat, rayakan, lebar..."
4152,frasa agama akan dihapus dalam peta jalan pend...,"[frasa, agama, akan, dihapus, dalam, peta, jal..."
4153,samsung menarik iklan senilai miliar dengan o...,"[samsung, menarik, iklan, senilai, miliar, den..."


# Stopword Removal

In [15]:
# Ambil stopwords standar Bahasa Indonesia
list_stopwords = set(stopwords.words('indonesian'))

In [16]:
# 2. Daftar Noise
noise_tambahan = [
    # Noise Teknis Website
    'advertisement', 'continue', 'scroll', 'content', 'read', 'share', 
    'copyright', 'rights', 'reserved', 'tonton', 'video', 'detik', 'com',
    'cnn', 'indonesia', 'jakarta', 'redaksi', 'klik', 'halaman', 'berita',
    'artikel', 'tag', 'bc', 'ya', 'reaksi', 'tulis', 'source', 'foto', 
    'gambar', 'url', 'link', 'app', 'android', 'ios', 'install', 'update',
    'newsletter', 'subscribe', 'headline', 'news', 'anchor', 'tutup', 'pilih',
    'with', 'to',
    
    # Noise Proses Klarifikasi (Turnbackhoax)
    'klaim', 'salah', 'narasi', 'judul', 'foto', 'video', 'unggah', 'posting', 
    'postingan', 'edar', 'hasil', 'periksa', 'fakta', 'kategori', 'konten', 
    'sesat', 'manipulasi', 'disinformasi', 'hoaks', 'hoax', 'lansir', 'kait', 
    'sebut', 'beber', 'nyata', 'penjelasan', 'kesimpulan', 'sumber', 'arsip',
    'cek', 'faktanya', 'benarnya', 'telusur', 'klarifikasi', 'tangkapan', 'layar',
    'screen', 'shot', 'cetak', 'beredar', 'pesan', 'berantai', 'grup', 'whatsapp'
]

In [17]:
# 3. Masukkan ke dalam list_stopwords utama
list_stopwords.update(noise_tambahan)

In [18]:
# 4. Definisi Fungsi 
def remove_stopwords(tokens):
    # Fungsi ini akan mengecek setiap kata apakah ada di dalam list_stopwords yang sudah diupdate tadi
    return [word for word in tokens if word not in list_stopwords]

In [20]:
# 5. Proses penyaringan
df['step_4_stopword'] = df['step_3_tokenizing'].apply(remove_stopwords)
print("\n--- STEP 4: STOPWORD REMOVAL SELESAI (Menghapus Kata Tidak Penting) ---")
display(df[['step_3_tokenizing', 'step_4_stopword']])


--- STEP 4: STOPWORD REMOVAL SELESAI (Menghapus Kata Tidak Penting) ---


,step_3_tokenizing,step_4_stopword
0,"[habiburokhman, soal, polisi, tembak, siswa, s...","[habiburokhman, polisi, tembak, siswa, diklaim..."
1,"[video, anak, anak, gaza, mendapat, kesempatan...","[anak, anak, gaza, kesempatan, bermain, pantai..."
2,"[video, hari, ini, ditemukan, oleh, relawan, a...","[ditemukan, relawan, anies, makassar, rencana,..."
3,"[anies, bongkar, misi, busuk, prabowo, usai, t...","[anies, bongkar, misi, busuk, prabowo, tom, le..."
4,"[video, demo, cirebon, pada, juni, akun, tikto...","[demo, cirebon, juni, akun, tiktok, roby, memp..."
...,...,...
4150,"[video, warga, israel, kabur, dengan, tangga, ...","[warga, israel, kabur, tangga, eskalator, iran..."
4151,"[puan, berkebaya, kuning, saat, rayakan, lebar...","[puan, berkebaya, kuning, rayakan, lebaran, me..."
4152,"[frasa, agama, akan, dihapus, dalam, peta, jal...","[frasa, agama, dihapus, peta, jalan, pendidika..."
4153,"[samsung, menarik, iklan, senilai, miliar, den...","[samsung, menarik, iklan, senilai, miliar, oli..."


# Stemming

In [21]:
# Inisialisasi
factory = StemmerFactory()
stemmer = factory.create_stemmer()

In [22]:
print("\n--- TAHAP 5: PROSES STEMMING SEDANG BERJALAN (SASTRAWI) ---")
# Kita gabungkan kembali list menjadi string sebelum masuk stemming agar Sastrawi bekerja lebih efisien
df['step_5_stemming'] = df['step_4_stopword'].progress_apply(lambda x: stemmer.stem(' '.join(x)))

print("\n--- STEP 5: STEMMING SELESAI (Mengubah ke Kata Dasar) ---")
display(df[['step_4_stopword', 'step_5_stemming']])


--- TAHAP 5: PROSES STEMMING SEDANG BERJALAN (SASTRAWI) ---


100%|██████████████████████████████████████████████████████████████████████████████| 4155/4155 [34:05<00:00,  2.03it/s]


--- STEP 5: STEMMING SELESAI (Mengubah ke Kata Dasar) ---


,step_4_stopword,step_5_stemming
0,"[habiburokhman, polisi, tembak, siswa, diklaim...",habiburokhman polisi tembak siswa klaim gangst...
1,"[anak, anak, gaza, kesempatan, bermain, pantai...",anak anak gaza sempat main pantai serang iran ...
2,"[ditemukan, relawan, anies, makassar, rencana,...",temu rawan anies makassar rencana curang milu ...
3,"[anies, bongkar, misi, busuk, prabowo, tom, le...",anies bongkar misi busuk prabowo tom lembong t...
4,"[demo, cirebon, juni, akun, tiktok, roby, memp...",demo cirebon juni akun tiktok roby memposting ...
...,...,...
4150,"[warga, israel, kabur, tangga, eskalator, iran...",warga israel kabur tangga eskalator iran rudal...
4151,"[puan, berkebaya, kuning, rayakan, lebaran, me...",puan kebaya kuning raya lebaran megawati ketua...
4152,"[frasa, agama, dihapus, peta, jalan, pendidika...",frasa agama hapus peta jalan didik kemendikbud...
4153,"[samsung, menarik, iklan, senilai, miliar, oli...",samsung tarik iklan nila miliar olimpiade pari...


# Finalisasi & Export

In [27]:
df['text_clean'] = df['step_5_stemming']

# Hapus data yang mungkin jadi kosong setelah dibersihkan
df.dropna(subset=['text_clean'], inplace=True)
df = df[df['text_clean'] != ""]

In [28]:
output_file = 'dataset_preprocessed_lengkap.xlsx'
df.to_excel(output_file, index=False)